In [ ]:
from blochstate1d_NEW import OLConstants
import numpy as np
from ga_tools import fitness_finder, sorted_fitnesses, fitness_error
from genetic_algorithm import full_genetic_algorithm
from ga_individual_maker import make_controls_fn1, OpticalLatticeHamiltonian
from blochstate1d_NEW import OLConstants, GroundBlochState
from errors_momentum_space import momentum_space_population, dot_product_error, get_error
from quevolutio.core.domain import QuantumHilbertSpace, TimeGrid
from typing import Optional, cast
from quevolutio.core.domain import QuantumHilbertSpace, TimeGrid
from quevolutio.core.aliases import (  # isort: skip
    RVector,
    GVector,
    RVectorSeq,
    CTensors,
    CSRMatrix,
)
from quevolutio.propagators.split_operator import SplitOperator
import time as time_module

In [ ]:
#initialise constants and parameters
constants = OLConstants()
N_individuals = OLConstants.N_individuals  # Number of individuals in the population
N_elements = OLConstants.N_elements      # Number of elements in the control function (e.g., number
sigma = 100 #standard deviation for the normal distribution to generate the coefficients from
mom_pop_des = constants.desired_mom_pop #desired momentum population from the constants file
mom_pop_des = np.array(mom_pop_des) #convert to numpy array for easier calculations
number_dead = 8 #number of individuals to remove each generation
N_test = 100
#Initialise many test coeffs. We will pick out the best 20 to run in the GA
test_A_coeffs = np.random.normal(0, sigma, (N_test, N_elements)) #generating the A coefficients from a normal distribution
test_B_coeffs = np.random.normal(0, sigma, (N_test, N_elements)) #generating the B coefficients from a normal distribution
test_omegas = np.random.uniform(0, 0.6, (N_test, N_elements)) #generating the frequencies from a uniform distribution between 0 and 0.6

In [ ]:
def tester(A_coeffs, B_coeffs, omegas, mom_pop_des):
    """
    Calculates the fitness of an individual by finding the error from the desired momentum population.
    """
    #Initialise the system
    constants: OLConstants = OLConstants()
    domain: QuantumHilbertSpace = QuantumHilbertSpace(
        num_dimensions=1,
        num_points=np.array([constants.num_pts]),
        position_bounds=np.array([[constants.lower_x_bound, constants.upper_x_bound]]),
        constants=constants,
    )
    initial_state = GroundBlochState().generate_bloch_state()
    state_initial: RVector = cast(
        RVector, domain.normalise_state(initial_state)
    )

    # Set up the Hamiltonian.
    hamiltonian: OpticalLatticeHamiltonian = OpticalLatticeHamiltonian(domain)

    # Set up the time domain.
    time_domain: TimeGrid = TimeGrid(time_min=0.0, time_max=OLConstants.T, num_points=10001)

    # Set up the propagator.
    propagator = SplitOperator(hamiltonian, time_domain)
    fitnesses_test = np.zeros(N_test)
    # Make the shaking function (controls function), for the given coefficients and frequencies
    for i in range(N_test):
        print(f"Making controls for individual {i+1}")
        A = A_coeffs[i, :]
        B = B_coeffs[i, :]
        omega = omegas[i, :]
        controls = make_controls_fn1(OLConstants.T, A, B, omega)
    
        
    # 2. Propagate the initial state (timed)
        
        print("Propagation Start")
        start_time: float = time_module.time()
        states: CTensors = propagator.propagate(
            state_initial, controls, diagnostics=False
            )
        final_time: float = time_module.time()
        print("Propagation Done")
        runtime = final_time - start_time
        # print(f"Runtime: {runtime:.2f} seconds")
        final_state = states[-1]

        #find momentum space population
        x_grid_spacing = GroundBlochState().x_grid[1] - GroundBlochState().x_grid[0]
        mom_pop = momentum_space_population(final_state, x_grid_spacing)

        #calculate fitness
        fitness = fitness_error(mom_pop,mom_pop_des)
        print(f"Fitness for individual {i+1}: {fitness:.6f}")
        fitnesses_test[i] = fitness
    
    return fitnesses_test, mom_pop

In [ ]:
test_fits, test_pop = tester(test_A_coeffs, test_B_coeffs, test_omegas, mom_pop_des)
sorted_test_fits, sorted_test_A, sorted_test_B, sorted_omegas = sorted_fitnesses(test_fits, test_A_coeffs, test_B_coeffs, test_omegas)
print(test_fits.shape)
print(sorted_test_A.shape)
#select best 20
A_coeffs = sorted_test_A[:OLConstants.N_individuals, :]
B_coeffs = sorted_test_B[:OLConstants.N_individuals, :]
omegas = sorted_omegas[:OLConstants.N_individuals, :]

In [ ]:
#This is the first run of the full genetic algorithm, which will return the best fitness of each generation in a list
bestfits, bestAs, bestBs, bestOmegas, final_mom_pop = full_genetic_algorithm(A_coeffs, B_coeffs, omegas, mom_pop_des, number_dead)
best_fit_run1 = np.min(bestfits)
print(f"Best fitness from run 1: {best_fit_run1}")

In [ ]:
import matplotlib.pyplot as plt
# Plotting the best fitness of each generation
plt.figure(figsize=(10, 6))
plt.plot(bestfits, marker='none')
plt.title('Best Fitness of Each Generation')
plt.xlabel('Generation')
plt.ylabel('Best Fitness')
plt.grid()
plt.show()

#saving the best fitnesses at each generation to a csv file
import pandas as pd
df = pd.DataFrame({'Best Fitness': bestfits})
df.to_csv('waste.csv', index=False)




In [ ]:
#plotting final momentum state and overlaying the desired momentum state
momentum_idx = np.arange(-10, 12, 2)

plt.figure(figsize=(10, 6))
plt.xticks(momentum_idx, [str(int(x)) for x in momentum_idx])

plt.bar(momentum_idx, final_mom_pop, alpha=1.0, label='Final Momentum Population', color = 'C0')
plt.bar(momentum_idx, mom_pop_des, alpha=1.0, label='Desired Momentum Population', fill = False, edgecolor='black', linewidth=3)
plt.title('Momentum State Population of Final State vs Desired State', fontsize = 16)
plt.xlabel('Momentum State (2 hk)', fontsize = 14)
plt.ylabel('Population', fontsize = 14)
plt.legend()
plt.grid()
plt.show()

In [ ]:
#determination of error
error = dot_product_error(final_mom_pop, mom_pop_des)
print(f"Error from desired momentum population: {error:.6f}")
#save the best coefficients and frequencies to text files
np.savetxt("bestA.npy", bestAs)
np.savetxt("bestB.npy", bestBs)
np.savetxt("bestOmega.npy", bestOmegas)

bestOmega1 = np.array(bestOmegas[0,:])
bestAs1 = np.array(bestAs[0,:])
bestBs1 = np.array(bestBs[0,:])
np.savez('randomsol.npz', A=bestAs, B=bestBs, omegas=bestOmegas) #CHANGE THIS TO WHAT YOU NEED
print(bestAs1.shape)
print(bestBs1.shape)
print(bestOmega1.shape)


In [ ]:
print("Best A coefficients:", bestAs1)
print("Best B coefficients:", bestBs1)
print("Best Omega frequencies:", bestOmega1)